In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def U(a):
    return np.array([[a, np.sqrt(1-a**2)], [np.sqrt(1-a**2), -a]])

def S(phi):
    return np.array([[np.exp(1j*phi), 0], [0, np.exp(-1j*phi)]])

def qsp_matrix(angles, a):
    matrix = S(angles[0])
    for i in range(1, len(angles)):
        matrix = matrix @ U(a) @ S(angles[i])
    return matrix

def coefficients(angles):
    a_values = np.array([0.3, 0.7, 0.9])
    P_a_values = np.array([qsp_matrix(angles, a)[0, 0] for a in a_values])
    A = np.array([[a**3, a] for a in a_values])
    coeffs = np.linalg.lstsq(A, P_a_values, rcond=None)[0]
    return coeffs

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    outs = [[float(np.real(elem)), float(np.imag(elem))] for elem in coeffs]
    return json.dumps(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol = 1e-4)

test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if not np.allclose(json.loads(output), json.loads(expected_output), rtol = 1e-4):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")


Running test case 0 with input '[0.5,0.8,1.0,1.0]'...
Correct!
Running test case 1 with input '[-0.20409113, -0.91173829, 0.91173829, 0.20409113]'...
Correct!
